# Практика 3/3: Google AI Studio — Gemini API

> **Курс «От нуля до своих агентов» — Модуль 1.5.**
> Третья из трёх практик. На этой задаче вы трогаете **только Google
> AI Studio**: достаёте ключ из Kaggle Secrets и шлёте запросы в
> `gemini-2.5-flash` через `google-genai` SDK. Никаких HF.
>
> Время: ~10 минут.

## Что делаем

Foundation-LLM — это «универсальный кубик» из ветки C карты ML. Тот же
вопрос (классификация настроения короткой фразы) можно решать им — без
обучения, без датасета, через промпт. Сравним с тем, что было на HF —
прицельно увидим, в чём foundation-модель лучше, а в чём избыточна.

Шаг с structured output (enum mode) важен сам по себе: без него LLM
норовит ответить «I think it's POSITIVE because...» — и весь
пайплайн ломается на парсинге.

## Шаг 1. Setup и ключ

In [ ]:
!pip install -q google-genai

In [ ]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
print("Ключ Gemini загружен, длина:", len(GOOGLE_API_KEY), "символов")

## Шаг 2. Первый запрос — sanity check

In [ ]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

resp = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Поздоровайся одним предложением на русском.",
)
print(resp.text)

## Шаг 3. Sentiment через промпт — наивная версия

Сначала — как «обычный программист сделал бы 2 года назад»: просим
модель ответить словом, парсим текст руками.

In [ ]:
from google.genai import types

# TODO: замените на свои 5 фраз (минимум одна — на русском, одна с подвохом)
phrases = [
    "TODO: позитивная фраза",
    "TODO: негативная фраза",
    "TODO: фраза с сарказмом",
    "TODO: что-нибудь на русском",
    "TODO: короткая фраза, 1-3 слова",
]

cfg_naive = types.GenerateContentConfig(
    system_instruction=(
        "You are a sentiment classifier. For each input, return EXACTLY "
        "one word: POSITIVE or NEGATIVE. No explanations."
    ),
    temperature=0,
    max_output_tokens=4,
)

for phrase in phrases:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        config=cfg_naive,
        contents=phrase,
    )
    label = resp.text.strip().upper().rstrip(".!?")
    print(f"{label:8s} | {phrase}")

## Шаг 4. Structured output — нормальная версия

Тот же запрос, но теперь говорим модели: «возвращай только значение из
этого enum'а, никакого свободного текста». Это надёжнее на порядок —
модель ломается куда реже.

In [ ]:
import enum


class Sentiment(enum.Enum):
    POSITIVE = "POSITIVE"
    NEUTRAL = "NEUTRAL"
    NEGATIVE = "NEGATIVE"


cfg_enum = types.GenerateContentConfig(
    response_mime_type="text/x.enum",
    response_schema=Sentiment,
    temperature=0,
)

results = []
for phrase in phrases:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        config=cfg_enum,
        contents=f"Classify sentiment of this text:\n{phrase}",
    )
    # response.parsed — это уже Python enum, не строка
    results.append(resp.parsed)
    print(f"{resp.parsed.value:8s} | {phrase}")

## Шаг 5. Меняем температуру и смотрим, что плывёт

**TODO:** запустите ячейку. На каких фразах модель «передумала» при
высокой температуре?

In [ ]:
cfg_hot = types.GenerateContentConfig(
    system_instruction="Return EXACTLY one word: POSITIVE or NEGATIVE.",
    temperature=2.0,
    max_output_tokens=4,
)

print("Температура 2.0 (5 запусков одной фразы):")
test_phrase = phrases[2]  # фраза с сарказмом
for _ in range(5):
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        config=cfg_hot,
        contents=test_phrase,
    )
    print(f"  {resp.text.strip()}")

## Шаг 6. Один абзац о наблюдении

**TODO:** одно-два предложения:

1. Где Gemini справился, а distilbert из практики 2 — нет (или наоборот)?
2. Что дал structured output по сравнению с наивным промптом? Когда без
   него можно было обойтись?

**TODO: ваше наблюдение здесь.**

## Сдача

`Save & Run All` → `Share Public` → ссылка в чат:
`[Модуль 1.5, практика AI Studio] {ссылка}`.

Все три практики сданы — поздравляю, инструментарий настроен и
проверен в бою. Дальше — Модуль 2 (карта ML-вселенной).